# 01 · Data Understanding
Structure, volume, keys and relationships of the nine raw datasets *before* cleaning.
> All data is synthetic (Faker + NumPy, seed 42). No real personal data is used.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.float_format", "{:,.2f}".format)
from src.data_cleaning.loader import load_all_processed, load_all_raw
from src.kpi.kpis import kpi_by, compute_all_kpis
from src.analytics import eda
d = load_all_processed()
t, f, o, c = d["tickets_enriched"], d["customer_feedback"], d["operations"], d["customers"]
VIS = ROOT / "visualizations"
def show(*names):
    for n in names: display(Image(filename=str(VIS / f"{n}.png"), width=820))

In [2]:
raw = load_all_raw()
pd.DataFrame({"rows": {k: len(v) for k, v in raw.items()}, "columns": {k: v.shape[1] for k, v in raw.items()}})

,rows,columns
customers,3230,9
departments,6,5
agents,60,9
support_tickets,24150,16
customer_feedback,12538,8
operations,14248,11
agent_performance,1984,10
business_costs,1296,7
calendar,1826,11


## Schema of each raw table

In [3]:
from src.data_validation.validators import profile
prof = pd.concat([profile(v, k) for k, v in raw.items()])
prof[prof.table.isin(["support_tickets", "customers", "customer_feedback"])]

,table,column,dtype,non_null,missing,missing_pct,distinct
0,customers,customer_id,str,3230,0,0.00,3200
1,customers,customer_name,str,3230,0,0.00,3089
2,customers,customer_segment,str,3230,0,0.00,8
3,customers,customer_region,str,3185,45,1.39,5
4,customers,customer_country,str,3230,0,0.00,21
5,customers,customer_type,str,3230,0,0.00,2
6,customers,registration_date,str,3230,0,0.00,1522
7,customers,customer_status,str,3230,0,0.00,3
8,customers,customer_lifetime_value,float64,3158,72,2.23,3122
0,support_tickets,ticket_id,str,24150,0,0.00,24000


## Sample records

In [4]:
raw["support_tickets"].head(5)

,ticket_id,customer_id,agent_id,department_id,ticket_created_date,first_response_date,resolved_date,ticket_category,ticket_subcategory,ticket_priority,ticket_status,support_channel,resolution_type,reopened_count,escalation_flag,sla_target_hours
0,TCK-000001,CUS-00934,AGT-046,DEP-05,2023-01-01 10:04:16,2023-01-02 05:43:43,2023-01-03 13:18:35.000000000,Returns & Refunds,Exchange Request,High,Closed,Social Media,Escalated Resolution,0,1,24
1,TCK-000002,CUS-01955,AGT-021,DEP-02,2023-01-01 11:57:22,2023-01-02 12:18:11,2023-01-03 02:13:02.000000000,Billing & Payment,Invoice Request,Medium,Closed,Email,Refund Issued,0,0,48
2,TCK-000003,CUS-01433,AGT-033,DEP-03,2023-01-01 12:07:40,2023-01-01 12:28:20,2023-01-08 15:19:25.000000000,Order Status,Missing Items,Low,Closed,Live Chat,Information Provided,0,0,72
3,TCK-000004,CUS-02103,AGT-018,DEP-02,2023-01-01 13:48:05,2023-01-01 13:58:49,NaN,Billing & Payment,Incorrect Charge,Low,Cancelled,Phone,NaN,0,0,72
4,TCK-000005,CUS-00868,AGT-043,DEP-05,2023-01-01 14:25:37,2023-01-02 05:50:30,2023-01-02 14:23:08.000000000,Returns & Refunds,Refund Status,High,Closed,Email,Refund Issued,0,0,24


In [5]:
raw["customers"].head(5)

,customer_id,customer_name,customer_segment,customer_region,customer_country,customer_type,registration_date,customer_status,customer_lifetime_value
0,CUS-00001,Tracy House,Consumer,Middle East & Africa,Saudi Arabia,Individual,2023-10-01,Inactive,488.96
1,CUS-00002,Carol Tucker,Consumer,Europe,France,Individual,2021-03-19,Inactive,"1,559.74"
2,CUS-00003,Hancock Inc,Enterprise,North America,Canada,Business,2023-03-15,Active,"31,687.31"
3,CUS-00004,Gray Group,Enterprise,Europe,Germany,Business,2022-09-25,Active,"37,648.11"
4,CUS-00005,House-Glover,Enterprise,Asia Pacific,Australia,Business,2025-06-09,Inactive,"23,243.14"


## Relationships (entity model)
```
customers 1─* support_tickets *─1 agents *─1 departments
support_tickets 1─0..1 customer_feedback
departments 1─* operations · 1─* business_costs
agents 1─* agent_performance (monthly)
calendar ─ conformed date dimension (2021-2025)
```

In [6]:
import json
manifest = json.loads((ROOT / "data/raw/_generation_manifest.json").read_text())
pd.Series({k: v for k, v in manifest.items() if isinstance(v, int)}, name="injected defects")

customers_missing_region               45
customers_missing_clv                  70
customers_alt_date_format              60
customers_bad_case_segment             40
customers_exact_duplicates             30
tickets_resolved_before_created        25
tickets_missing_first_response         90
tickets_missing_channel               120
tickets_channel_label_variants        400
tickets_priority_upper_case           300
tickets_wrong_sla_target              180
tickets_orphan_agent                   15
tickets_exact_duplicates              150
feedback_out_of_range_score            40
feedback_missing_score                110
feedback_customer_mismatch_max         25
feedback_orphan_ticket                 12
feedback_exact_duplicates              60
operations_missing_processing_time     70
operations_negative_cost               12
operations_cost_outliers               18
operations_success_failed_mismatch     20
costs_total_mismatch                   25
costs_missing_indirect            

## Observations
* Tickets cover **Jan 2023 - Dec 2025** (3 years); customers registered from 2021.
* The raw data deliberately contains duplicates, mixed date formats, inconsistent labels, impossible timestamps, out-of-range scores, orphan keys and cost outliers. The manifest above records exactly what was injected so the cleaning stage can be verified against it (see notebook 02 and `tests/test_cleaning.py`).